In [1]:
%load_ext autoreload
%autoreload 2

In [59]:
%matplotlib qt

In [2]:
from pathlib import Path
import pandas as pd

In [3]:
from algorithm import PGDAlgorithm
import numpy as np

In [4]:
DATA_DIR = Path("/home/sean/code/study-hard/controls/online_optimization/project/datasets")
ALL_LMPS_FILENAME = DATA_DIR / "july_12_2023_all_lmps.csv"
YEAR_LMP_FILENAME = "lmp_for_year.csv"

In [5]:
lmp = pd.read_csv(DATA_DIR / YEAR_LMP_FILENAME).set_index("Time")

In [6]:
lmp

,Unnamed: 0,Market,Location,Location Type,LMP,Energy,Congestion,Loss
Time,,,,,,,,
2023-01-01 00:00:00-08:00,0,REAL_TIME_15_MIN,AVENAL_6_GN001,NaN,114.91000,114.91000,0.0,0.00000
2023-01-01 00:00:00-08:00,1,REAL_TIME_15_MIN,CATALYST_7_N002,NaN,114.91000,114.91000,0.0,0.00000
2023-01-01 00:00:00-08:00,2,REAL_TIME_15_MIN,PTHLGEN_7_N002,NaN,114.91000,114.91000,0.0,0.00000
2023-01-01 00:00:00-08:00,3,REAL_TIME_15_MIN,SANRAFL_1_N007,NaN,114.91000,114.91000,0.0,0.00000
2023-01-01 00:15:00-08:00,4,REAL_TIME_15_MIN,AVENAL_6_GN001,NaN,114.41000,114.41000,0.0,0.00000
...,...,...,...,...,...,...,...,...
2023-12-14 23:30:00-08:00,379,REAL_TIME_15_MIN,SANRAFL_1_N007,NaN,56.59206,55.78870,0.0,0.80336
2023-12-14 23:45:00-08:00,380,REAL_TIME_15_MIN,AVENAL_6_GN001,NaN,51.85445,52.27263,0.0,-0.41818
2023-12-14 23:45:00-08:00,381,REAL_TIME_15_MIN,CATALYST_7_N002,NaN,54.12308,52.27263,0.0,1.85045


In [7]:
locations = lmp.Location.unique()

In [8]:
loc = "PTHLGEN_7_N002"

In [9]:
lmp_loc = lmp[lmp.Location == loc]

In [10]:
lmp_loc.shape

(33408, 8)

In [11]:
lmp.shape[0] / 4

33408.0

In [12]:
P_max = 300  # MW
E_max = P_max * 4  # MWh, 4hr battery
depth_of_discharge = 0.1
E_min = E_max * depth_of_discharge
dt = 15 / 60  # hr


In [13]:
lifetime = 20  # year
cost_cap = P_max * 300 + E_max * 350  # $300/MW + $350/MWh
n_cycles_lifetime = 365 * lifetime  # 1 cycle/day * lifetime (days) = num_cycles
T_E = E_max / P_max  # battery duration (hr)
# Battery degradation constant
gamma = 2 * cost_cap * dt / (n_cycles_lifetime * T_E * P_max**2)  # $/MW^2

In [14]:
print("Degradation constant ($/KW^2):", gamma * 1e6)

Degradation constant ($/KW^2): 97.03196347031964


In [52]:
gamma * 

9.703196347031964e-05

In [15]:
# Time horizon (steps in 1day)
T = int(24 / dt)

In [120]:
results = {}

In [164]:
alpha = 1

In [171]:
alg = PGDAlgorithm(
    gamma * 1000, E_min, E_max, dt, T, alpha, max_iter=int(1e5))

In [124]:
prices = np.array(lmp_loc.LMP)
soc, u_hat = alg.simulate(prices[:500])

Step:  0
Step:  1
Step:  2
Step:  3
Step:  4
Step:  5
Step:  6
Step:  7
Step:  8
Step:  9
Step:  10
Step:  11
Step:  12
Step:  13
Step:  14
Step:  15
Step:  16
Step:  17
Step:  18
Step:  19
Step:  20
Step:  21
Step:  22
Step:  23
Step:  24
Step:  25
Step:  26
Step:  27
Step:  28
Step:  29
Step:  30
Step:  31
Step:  32
Step:  33
Step:  34
Step:  35
Step:  36
Step:  37
Step:  38
Step:  39
Step:  40
Step:  41
Step:  42
Step:  43
Step:  44
Step:  45
Step:  46
Step:  47
Step:  48
Step:  49
Step:  50
Step:  51
Step:  52
Step:  53
Step:  54
Step:  55
Step:  56
Step:  57
Step:  58
Step:  59
Step:  60
Step:  61
Step:  62
Step:  63
Step:  64
Step:  65
Step:  66
Step:  67
Step:  68
Step:  69
Step:  70
Step:  71
Step:  72
Step:  73
Step:  74
Step:  75
Step:  76
Step:  77
Step:  78
Step:  79
Step:  80
Step:  81
Step:  82
Step:  83
Step:  84
Step:  85
Step:  86
Step:  87
Step:  88
Step:  89
Step:  90
Step:  91
Step:  92
Step:  93
Step:  94
Step:  95
Step:  96
Step:  97
Step:  98
Step:  99
Step:  100

In [121]:
results[1] = (soc, u_hat)

In [113]:
len(u_hat)

405

In [114]:
import matplotlib.pyplot as plt

In [130]:
soc, u_hat = results[1]

In [139]:
t_sim = np.arange(0, dt * 500, dt)

In [132]:
u_hat = np.array(u_hat)

In [133]:
u_hat.shape

(405, 96)

In [142]:
len(soc)

405

In [145]:
u_chosen = np.r_[u_hat[:, 0], u_hat[-1, 1:]]
x_chosen = np.r_[soc, soc[-1] + alg.Gamma @ u_hat[-1]][:-1]

In [150]:
x_pred = [
    x + alg.Gamma @ u
    for x, u in zip(soc[:-1], u_hat[1:])
]

In [151]:
t_sim.shape

(500,)

In [152]:
x_chosen.shape

(500,)

In [ ]:
sell_price = gamma / (delta_t * 

In [156]:
from algorithm import MarketESSCost

In [207]:
alg.alpha = 0.1

In [208]:
cost = [
    MarketESSCost(alg.gamma, alg.delta_t, prices[i:i+T]).eval(u)
    for i, u in enumerate(u_hat)]

In [211]:
u_stars = []
all_optimal_costs = []
for i, u0 in enumerate(u_hat):
    if i % 30 != 0:
        continue
    print(f"Computing for {i}")
    _, u_single_price = alg.simulate(
        prices[i:i+T], u0=u0,n_steps=10, use_same_price=True)
    u_stars.append(np.array(u_single_price))
    optimal_c = []
    for u in u_single_price:
        optimal_c.append(
            MarketESSCost(alg.gamma, alg.delta_t, prices[i:i+T]).eval(
                u)
        )
    all_optimal_costs.append(np.array(optimal_c))
    

Computing for 0
Computing for 30
Computing for 60
Computing for 90
Computing for 120
Computing for 150
Computing for 180
Computing for 210
Computing for 240
Computing for 270
Computing for 300
Computing for 330
Computing for 360
Computing for 390


In [201]:
len(all_optimal_costs)

14

In [213]:
plt.figure()
plt.plot(np.array(all_optimal_costs).T)
plt.xlabel("Iteration Index")
plt.ylabel("Cost")
plt.legend([
    f"t={i * dt}" for i in range(0, 30 * len(all_optimal_costs), 30)
])
plt.title("Computing Cost of u*")


Text(0.5, 1.0, 'Computing Cost of u*')

In [189]:
optimal_cost = []
for i, u0 in enumerate(u_hat):
    if i != 200:
        continue
#     if i % 5 != 0:
#         continue
    print(f"Computing for {i}")
    _, u_star = alg.simulate(
        prices[i:i+T], u0=u0,n_steps=20, use_same_price=True)
    u_stars.append(u_single_price[-1])
    for u in u_star:
        optimal_cost.append(
            MarketESSCost(alg.gamma, alg.delta_t, prices[i:i+T]).eval(
                u)
        )
    

Computing for 200


In [191]:
plt.figure()
plt.plot(optimal_cost)

In [179]:
t_optimal_cost = np.arange(0, dt * u_hat.shape[0], dt * 5)

In [181]:
len(t_optimal_cost)

81

In [180]:
len(optimal_cost)

81

In [187]:
_, ax = plt.subplots(2, 2, sharex=True)
ax[0,0].plot(t_sim, u_chosen, label="u[0] (Executed control)")
ax[1,0].plot(t_sim, x_chosen, label="SoC (Executed)")
for i, (u, x) in enumerate(zip(u_hat[1:], x_pred[:-1])):
    if i % 100 == 0:
        ax[0,0].plot(t_sim[i:i+T], u, label=f"u_hat (t={t_sim[i]:.2f}) (prediction)")
        ax[1,0].plot(t_sim[i:i+T], x, label=f"x_hat (t={t_sim[i]:.2f}) (prediction)")
# ax[0].legend()
# for i, x in enumerate(x_pred):
#     if i % 100 == 0:
#         ax[1].plot(i, x)
ax[0,0].set_title("Battery Power")
ax[1,0].set_title("Battery SoC")
ax[0,1].set_title(f"LMP at node {loc}")
ax[1,1].set_title(f"Cost function (Phi_t(u), Phi_t(u*))")

for i in range(2):
    for j in range(2):
        ax[i,j].set_xlabel("Time [hr]")
ax[0, 0].set_ylabel("Power [MW]")
ax[1, 0].set_ylabel("SoC [MWh]")
ax[0, 1].set_ylabel("LMP [$/MW]")
ax[1, 1].set_ylabel("Cost (Profit) [$]")
ax[0, 0].set_ylabel("Power [MW]")
ax[1,0].axhline(y=E_min, color="red", label="Min SoC")
ax[1,0].axhline(y=E_max, color="green", label="Max SoC")
ax[0,0].legend()
ax[1,0].legend()

ax[0,1].plot(t_sim, prices[:500])
ax[1,1].plot(t_sim[:len(cost)], cost, label="Cost")
ax[1,1].plot(t_optimal_cost, optimal_cost, label="Cost of u*")
ax[1,1].legend()



In [82]:
i = len(x_pred)
x0 = soc[-1]
u0 = u_hat[-1]
p = prices[i:i + T]
u_next = alg.algorithmic_map(x0, u0, p, project=False)

In [83]:
_, ax = plt.subplots(2, 1)
ax[0].plot(u_next)
ax[1].plot(x0 + alg.Gamma @ u0)
ax[1].axhline(y=E_min, color="red", label="Min SoC")
ax[1].axhline(y=E_max, color="green", label="Max SoC")

In [ ]:
plt.figure()


In [ ]:
plt.plot()

In [12]:
lmp.pivot(columns="Location")

ValueError: Index contains duplicate entries, cannot reshape